# 🧪 Sesión 03 · Headers, Content-Type y Negociación de Contenido

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo sabe el servidor qué formato le estás enviando?

---

## 🖥️ Así se verá el resultado final

```http
$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: application/json" \
       -H "Accept: application/json" \
       -d '{"titulo":"Rayuela","anio":1963}'
HTTP/1.1 201 Created
content-type: application/json
{"id":1,"titulo":"Rayuela","anio":1963}

$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: text/plain" \
       -d 'Rayuela'
HTTP/1.1 415 Unsupported Media Type

$ curl http://127.0.0.1:8000/api/libros \
       -H "Accept: application/xml"
HTTP/1.1 406 Not Acceptable
```

_El navegador envía Accept por ti; una herramienta como curl te obliga a ser explícito._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Leer y emitir encabezados HTTP correctamente.
- Diferenciar Content-Type (lo que envío) de Accept (lo que acepto).
- Rechazar formatos no soportados con 415 y 406.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] POST válido en JSON → HTTP 201
- [ ] POST en texto plano se rechaza con 415 → HTTP 415
- [ ] GET que pide XML se rechaza con 406 → HTTP 406
- [ ] GET que acepta JSON responde 200 → HTTP 200

## 🧩 El reto

Implementa la creación y el listado de libros, pero haz que la API negocie el formato de entrada y de salida.

**Requisitos del contrato**

- POST /api/libros solo acepta Content-Type: application/json; si no, responde 415.
- GET /api/libros solo responde si el cliente acepta application/json; si pide XML, responde 406.
- Las respuestas correctas llevan Content-Type: application/json.
- Un POST válido responde 201 con el libro creado.

> **Pista conceptual:** Puedes inspeccionar `request.headers` dentro de una dependencia o middleware antes de que se procese el cuerpo. El formato no se adivina: se declara.

## 📦 Entregable

Cuaderno con los cuatro casos en verde y bitácora `auditorias/auditoria_sesion03.md`.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [2]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [3]:

%%writefile app.py
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI(title="Tu API negociará el formato exactamente así")

libros = []
siguiente_id = 1

@app.post("/api/libros")
async def crear_libro(request: Request):
    global siguiente_id

    content_type = request.headers.get("content-type", "")
    if content_type.split(";")[0].strip() != "application/json":
        return JSONResponse(
            status_code=415,
            content={"error": "Formato no soportado"}
        )

    datos = await request.json()

    libro = {
        "id": siguiente_id,
        "titulo": datos["titulo"],
        "anio": datos["anio"]
    }
    libros.append(libro)
    siguiente_id += 1

    # ERROR INTENCIONAL: debería devolver 201
    return JSONResponse(status_code=200, content=libro)

@app.get("/api/libros")
async def listar_libros(request: Request):
    accept = request.headers.get("accept", "")

    if "application/xml" in accept:
        return JSONResponse(
            status_code=406,
            content={"error": "Formato de respuesta no aceptado"}
        )

    return JSONResponse(
        status_code=200,
        content=libros,
        media_type="application/json"
    )



Writing app.py


In [5]:

%%writefile app.py
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI(title="Tu API negociará el formato exactamente así")

libros = []
siguiente_id = 1


@app.post("/api/libros")
async def crear_libro(request: Request):
    global siguiente_id

    # Validar el formato de entrada
    content_type = request.headers.get("content-type", "")
    content_type = content_type.split(";")[0].strip().lower()

    if content_type != "application/json":
        return JSONResponse(
            status_code=415,
            content={"error": "Formato no soportado"}
        )

    # Leer y validar el JSON
    try:
        datos = await request.json()
    except Exception:
        return JSONResponse(
            status_code=400,
            content={"error": "JSON inválido"}
        )

    if not isinstance(datos, dict):
        return JSONResponse(
            status_code=400,
            content={"error": "Se esperaba un objeto JSON"}
        )

    if "titulo" not in datos or "anio" not in datos:
        return JSONResponse(
            status_code=400,
            content={"error": "Faltan titulo o anio"}
        )

    # Crear y guardar el libro
    libro = {
        "id": siguiente_id,
        "titulo": datos["titulo"],
        "anio": datos["anio"]
    }

    libros.append(libro)
    siguiente_id += 1

    # Responder con 201 Created
    return JSONResponse(
        status_code=201,
        content=libro,
        media_type="application/json"
    )


@app.get("/api/libros")
async def listar_libros(request: Request):
    # Validar el formato que acepta el cliente
    accept = request.headers.get("accept", "")
    formatos = [
        formato.split(";")[0].strip().lower()
        for formato in accept.split(",")
    ]

    acepta_json = any(
        formato in (
            "application/json",
            "application/*",
            "*/*"
        )
        for formato in formatos
    )

    if not acepta_json:
        return JSONResponse(
            status_code=406,
            content={"error": "Solo se responde en JSON"}
        )

    # Responder con 200 OK
    return JSONResponse(
        status_code=200,
        content=libros,
        media_type="application/json"
    )


Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Estoy trabajando con Python y FastAPI para desarrollar una API web. La actividad se realiza en un cuaderno Jupyter o en un entorno compatible, utilizando HTTP para probar las solicitudes y respuestas. La API debe cumplir con los requisitos establecidos por el docente. |
| **Rol** | La IA debe asumir el papel de un desarrollador web con conocimientos de Python, FastAPI y protocolos HTTP, que me ayude a implementar y revisar el funcionamiento de la API.. |
| **Especificación** | La API tendrá las rutas POST /api/libros y GET /api/libros. Debe aceptar JSON, responder 201 al crear un libro, 415 si el formato enviado no es compatible, 406 si se solicita XML y 200 al consultar libros en JSON. |
| **Output** | Código completo de app.py, con comentarios e instrucciones para realizar las cuatro pruebas y verificar que cumpla con los requisitos de la actividad. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [6]:
from verificar import probar_http

casos = [{'descripcion': 'POST válido en JSON',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'headers': {'Content-Type': 'application/json', 'Accept': 'application/json'},
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela', 'anio': 1963},
  'headers_contiene': {'content-type': 'application/json'}},
 {'descripcion': 'POST en texto plano se rechaza con 415',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'content': 'Rayuela',
  'headers': {'Content-Type': 'text/plain'},
  'estado': 415},
 {'descripcion': 'GET que pide XML se rechaza con 406',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'headers': {'Accept': 'application/xml'},
  'estado': 406},
 {'descripcion': 'GET que acepta JSON responde 200',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'headers': {'Accept': 'application/json'},
  'estado': 200,
  'headers_contiene': {'content-type': 'application/json'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. POST válido en JSON
✅ 02. POST en texto plano se rechaza con 415
✅ 03. GET que pide XML se rechaza con 406
✅ 04. GET que acepta JSON responde 200
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion03.md` a `auditorias/auditoria_sesion03.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?R.- El resultado que más me costó fue lograr que la API respondiera con los códigos HTTP correctos, porque tuve que revisar los encabezados Content-Type y Accept para distinguir el formato enviado del formato solicitado.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?R.- Decidí guardar los libros en una lista de Python y asignarles un ID automáticamente para poder identificarlos y consultarlos después de crearlos.
   - ¿Qué error detectó la verificación que tú no habías previsto?R.- La verificación me ayudó a identificar que el método POST respondía con 200 OK en lugar de 201 Created, aunque el libro se había creado correctamente. Tuve que corregir el código para cumplir con el contrato.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
